# WorldSim Contrastive DPO
Gemini Flash teacher → contrastive pairs → DPO training → evaluation


## 1. Environment Setup


In [ ]:
import json
import os
import random
import re
import shutil
import subprocess
import sys
import time
from collections import Counter, defaultdict
from datetime import UTC, datetime
from pathlib import Path

import yaml

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "config" / "generation.yaml").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

from dotenv import load_dotenv
load_dotenv(REPO_ROOT / ".env")

OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY")
assert OPENROUTER_API_KEY, "OPENROUTER_API_KEY not set"

TEACHER_MODEL = "google/gemini-2.5-flash"
API_BASE = "https://openrouter.ai/api/v1/chat/completions"
GGUF_DIR = REPO_ROOT / "artifacts" / "gguf"
LLAMA_SERVER = REPO_ROOT / "tools" / "llama.cpp" / "build" / "bin" / "llama-server"
LLAMA_QUANTIZE = REPO_ROOT / "tools" / "llama.cpp" / "build" / "bin" / "llama-quantize"
CONVERT_SCRIPT = REPO_ROOT / "tools" / "llama.cpp" / "convert_hf_to_gguf.py"
CONFIG_DIR = REPO_ROOT / "config"
CONTRASTIVE_DIR = REPO_ROOT / "data" / "rl" / "contrastive_dpo"
CONTRASTIVE_DIR.mkdir(parents=True, exist_ok=True)

from scripts.common import read_jsonl, write_jsonl
from scripts.reward_functions import combined_reward

generation = yaml.safe_load((CONFIG_DIR / "generation.yaml").read_text(encoding="utf-8"))
situations = yaml.safe_load((CONFIG_DIR / "situations.yaml").read_text(encoding="utf-8"))["situations"]
emotions = yaml.safe_load((CONFIG_DIR / "emotions.yaml").read_text(encoding="utf-8"))["emotions"]
temperaments = generation["temperaments"]
worlds = generation["worlds"]
names = generation["names"]

temp_by_id = {item["id"]: item for item in temperaments}
world_by_id = {item["id"]: item for item in worlds}
default_world = world_by_id[generation["v31_context"]["default_world"]]

CONTRASTIVE_PAIRS = [
    ("choleric", "melancholic"),
    ("sanguine", "phlegmatic"),
    ("choleric", "phlegmatic"),
    ("sanguine", "melancholic"),
]

TEACHER_PROMPT_TEMPLATE = (REPO_ROOT / "prompts" / "teacher" / "task_e.txt").read_text(encoding="utf-8")
L3_EN_SYSTEM = (REPO_ROOT / "prompts" / "training" / "layer3_en_system.txt").read_text(encoding="utf-8").strip()

print(f"Teacher model: {TEACHER_MODEL}")
print(f"Situations: {len(situations)}")
print(f"Temperaments: {len(temp_by_id)}")
print(f"Contrastive pairs: {len(CONTRASTIVE_PAIRS)}")
print(f"Teacher prompt vars loaded: {'scenario_desc' in TEACHER_PROMPT_TEMPLATE and 'action_options' in TEACHER_PROMPT_TEMPLATE}")


## 2. Generate Contrastive Outputs


In [ ]:
import urllib.request

N_ROUNDS = 10
CONTRASTIVE_OUTPUTS_PATH = CONTRASTIVE_DIR / "teacher_contrastive_outputs.jsonl"
rng = random.Random(42)


def action_options_line(action_options):
    return " ".join(f"{idx}:{option}" for idx, option in enumerate(action_options))


def teacher_action_options_line(action_options):
    return "\n".join(f"{idx}:{option}" for idx, option in enumerate(action_options))


def call_teacher(system, user, max_retries=3):
    payload = {
        "model": TEACHER_MODEL,
        "messages": [
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        "temperature": 0.4,
        "max_tokens": 256,
        "response_format": {"type": "json_object"},
    }
    req = urllib.request.Request(
        API_BASE,
        data=json.dumps(payload).encode(),
        headers={
            "Authorization": f"Bearer {OPENROUTER_API_KEY}",
            "Content-Type": "application/json",
            "HTTP-Referer": generation["provider"]["request_headers"]["referer"],
            "X-Title": generation["provider"]["request_headers"]["title"],
        },
    )
    last_error = None
    for attempt in range(max_retries):
        try:
            with urllib.request.urlopen(req, timeout=90) as resp:
                data = json.loads(resp.read())
            raw = data["choices"][0]["message"]["content"]
            try:
                parsed = json.loads(raw)
            except json.JSONDecodeError:
                parsed = json.loads(re.search(r"\{.*\}", raw, flags=re.DOTALL).group(0))
            return raw, parsed, data.get("usage", {})
        except Exception as exc:
            last_error = exc
            time.sleep((attempt + 1) * 1.5)
    raise RuntimeError(f"Teacher call failed after retries: {last_error}")


def build_task_e_prompt(situation, temperament, name, emotion, stress, variant):
    world = default_world
    return TEACHER_PROMPT_TEMPLATE.format(
        tci_ns=temperament["tci"]["NS"],
        tci_ha=temperament["tci"]["HA"],
        tci_rd=temperament["tci"]["RD"],
        tci_p=temperament["tci"]["P"],
        temperament_id=temperament["id"],
        temperament_name=temperament["ko"],
        temperament_keywords=", ".join(temperament.get("keywords", [])),
        name=name,
        personality_keywords=", ".join(temperament.get("keywords", [])),
        emotion_name=emotion["id"],
        emotion_intensity=emotion.get("intensities", [0.7])[variant % len(emotion.get("intensities", [0.7]))],
        stress=stress,
        scenario_desc=situation["desc"],
        world_id=world["id"],
        world_desc=world["desc"],
        world_vocab_additions=", ".join(world.get("vocab_additions", [])),
        action_options=teacher_action_options_line(situation["action_options"]),
        variant=variant,
    )


def build_student_prompt(situation, temperament, name, emotion, stress):
    intensity = emotion.get("intensities", [0.7])[0]
    return (
        f"[TASK] E\n[TEMP]\nNS={temperament['tci']['NS']} HA={temperament['tci']['HA']} RD={temperament['tci']['RD']} P={temperament['tci']['P']} type={temperament['id']}\n"
        f"[PERSONALITY] {', '.join(temperament.get('keywords', []))}\n"
        f"[EMOTION] {emotion['id']}:{intensity}\n"
        f"[STRESS] {stress}\n"
        f"[SITUATION] {situation['desc']}\n"
        f"[OPTIONS]\n{action_options_line(situation['action_options'])}\n"
        "[OUTPUT FORMAT]\n"
        '{"action_id": number, "confidence": 0.0-1.0, "hint": "English sentence", "personality_reasoning": "TCI axis", "temperament_factor": "snake_case"}'
    )

teacher_system = "You are a structured behavioral teacher. Return JSON only. Select the most temperament-consistent action for the scenario."
usage_totals = defaultdict(int)
written = 0
with CONTRASTIVE_OUTPUTS_PATH.open("w", encoding="utf-8") as handle:
    for round_idx in range(N_ROUNDS):
        for situation in situations:
            if not situation.get("action_options"):
                continue
            for pair_idx, (left_id, right_id) in enumerate(CONTRASTIVE_PAIRS):
                left_temp = temp_by_id[left_id]
                right_temp = temp_by_id[right_id]
                name = names[(round_idx + pair_idx) % len(names)]
                emotion = emotions[(round_idx + pair_idx) % len(emotions)]
                stress = round(rng.uniform(0.2, 0.9), 1)
                teacher_prompt_left = build_task_e_prompt(situation, left_temp, name, emotion, stress, round_idx)
                teacher_prompt_right = build_task_e_prompt(situation, right_temp, name, emotion, stress, round_idx)
                student_prompt_left = build_student_prompt(situation, left_temp, name, emotion, stress)
                student_prompt_right = build_student_prompt(situation, right_temp, name, emotion, stress)
                raw_left, parsed_left, usage_left = call_teacher(teacher_system, teacher_prompt_left)
                raw_right, parsed_right, usage_right = call_teacher(teacher_system, teacher_prompt_right)
                for key in ("prompt_tokens", "completion_tokens", "total_tokens"):
                    usage_totals[key] += int(usage_left.get(key, 0)) + int(usage_right.get(key, 0))
                row = {
                    "round": round_idx,
                    "situation_id": situation["id"],
                    "pair": [left_id, right_id],
                    "emotion": emotion["id"],
                    "stress": stress,
                    "left": {
                        "temperament_id": left_id,
                        "student_prompt": student_prompt_left,
                        "teacher_prompt": teacher_prompt_left,
                        "raw": raw_left,
                        "parsed": parsed_left,
                    },
                    "right": {
                        "temperament_id": right_id,
                        "student_prompt": student_prompt_right,
                        "teacher_prompt": teacher_prompt_right,
                        "raw": raw_right,
                        "parsed": parsed_right,
                    },
                }
                handle.write(json.dumps(row, ensure_ascii=False) + "\n")
                written += 1
                if written % 50 == 0:
                    print(f"  [{written}] contrastive scenarios generated", flush=True)

blended_rate = 0.375 / 1_000_000
estimated_cost = usage_totals["total_tokens"] * blended_rate
print(f"Generated scenarios: {written}")
print(f"Usage: {dict(usage_totals)}")
print(f"Estimated cost: ${estimated_cost:.2f}")
print(f"Saved: {CONTRASTIVE_OUTPUTS_PATH}")


## 3. Build DPO Pairs


In [ ]:
contrastive_rows = read_jsonl(CONTRASTIVE_OUTPUTS_PATH)
DPO_PAIRS_PATH = CONTRASTIVE_DIR / "contrastive_dpo_pairs.jsonl"

pairs = []
gap_distribution = []
skipped_same_action = 0
skipped_small_gap = 0

for row in contrastive_rows:
    left = row["left"]
    right = row["right"]
    parsed_left = left.get("parsed") or {}
    parsed_right = right.get("parsed") or {}
    if "action_id" not in parsed_left or "action_id" not in parsed_right:
        continue
    if parsed_left["action_id"] == parsed_right["action_id"]:
        skipped_same_action += 1
        continue

    left_raw = json.dumps(parsed_left, ensure_ascii=False)
    right_raw = json.dumps(parsed_right, ensure_ascii=False)

    score_left_on_left = combined_reward(left_raw, left["student_prompt"], config_dir=CONFIG_DIR)
    score_right_on_left = combined_reward(right_raw, left["student_prompt"], config_dir=CONFIG_DIR)
    gap_left = score_left_on_left["total"] - score_right_on_left["total"]
    if gap_left > 0.05:
        pairs.append({
            "prompt": [{"role": "system", "content": L3_EN_SYSTEM}, {"role": "user", "content": left["student_prompt"]}],
            "chosen": left_raw,
            "rejected": right_raw,
            "task": "E",
            "pair_id": f"{row['situation_id']}-{row['round']}-left",
            "reward_gap": round(gap_left, 4),
        })
        gap_distribution.append(gap_left)
    else:
        skipped_small_gap += 1

    score_right_on_right = combined_reward(right_raw, right["student_prompt"], config_dir=CONFIG_DIR)
    score_left_on_right = combined_reward(left_raw, right["student_prompt"], config_dir=CONFIG_DIR)
    gap_right = score_right_on_right["total"] - score_left_on_right["total"]
    if gap_right > 0.05:
        pairs.append({
            "prompt": [{"role": "system", "content": L3_EN_SYSTEM}, {"role": "user", "content": right["student_prompt"]}],
            "chosen": right_raw,
            "rejected": left_raw,
            "task": "E",
            "pair_id": f"{row['situation_id']}-{row['round']}-right",
            "reward_gap": round(gap_right, 4),
        })
        gap_distribution.append(gap_right)
    else:
        skipped_small_gap += 1

write_jsonl(DPO_PAIRS_PATH, pairs)
print(f"Contrastive scenarios: {len(contrastive_rows)}")
print(f"DPO pairs: {len(pairs)}")
print(f"Skipped same action: {skipped_same_action}")
print(f"Skipped small gap: {skipped_small_gap}")
print(f"Average reward gap: {sum(gap_distribution)/len(gap_distribution):.3f}" if gap_distribution else 'Average reward gap: n/a')
print(f"Saved: {DPO_PAIRS_PATH}")


## 4. Reward Gap Analysis


In [ ]:
import statistics

print('=== Reward Gap Analysis ===')
if gap_distribution:
    print(f"Pairs: {len(gap_distribution)}")
    print(f"mean={statistics.mean(gap_distribution):.3f}")
    print(f"stdev={statistics.stdev(gap_distribution):.3f}" if len(gap_distribution) > 1 else 'stdev=0.000')
    print(f"min={min(gap_distribution):.3f} max={max(gap_distribution):.3f}")
    print('Histogram:')
    buckets = [0] * 10
    for gap in gap_distribution:
        idx = min(int(gap * 10), 9)
        buckets[idx] += 1
    scale = max(1, max(buckets) // 40) if buckets else 1
    for idx, count in enumerate(buckets):
        bar = '█' * (count // scale)
        print(f"  {idx/10:.1f}-{(idx+1)/10:.1f}: {bar} ({count})")
print('Best-of-N failed around 63 pairs / 4000 outputs; contrastive target is 400+ pairs.')


## 5. DPO Training


In [ ]:
import torch
from datasets import Dataset
from peft import LoraConfig, PeftModel, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import DPOConfig, DPOTrainer

SFT_HF_2B = "Qwen/Qwen3.5-2B-Base"
sft_root = REPO_ROOT / "outputs" / "baseline" / "worldsim-v31-2b"
sft_runs = sorted(sft_root.glob('run-*'))
assert sft_runs, f"No SFT runs found at {sft_root}"
latest_sft = sft_runs[-1]
sft_adapter_dir = latest_sft / 'adapter'
sft_merged_dir = latest_sft / 'merged_bf16'

print(f"Using SFT run: {latest_sft.name}")
base_source = sft_merged_dir if (sft_merged_dir / 'config.json').exists() else SFT_HF_2B
model = AutoModelForCausalLM.from_pretrained(str(base_source), torch_dtype=torch.bfloat16, device_map='auto')
tokenizer = AutoTokenizer.from_pretrained(str(base_source))
if base_source == SFT_HF_2B:
    model = PeftModel.from_pretrained(model, str(sft_adapter_dir)).merge_and_unload()

lora_config = LoraConfig(
    r=64,
    lora_alpha=128,
    lora_dropout=0.05,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
)
model = get_peft_model(model, lora_config)

pairs = read_jsonl(DPO_PAIRS_PATH)
formatted = []
for pair in pairs:
    prompt_text = "\n".join(message['content'] for message in pair['prompt'])
    formatted.append({"prompt": prompt_text, "chosen": pair['chosen'], "rejected": pair['rejected']})

dataset = Dataset.from_list(formatted)
split = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split['train']
eval_dataset = split['test']

RUN_ID = datetime.now(UTC).strftime('run-%Y%m%dT%H%M%SZ')
DPO_OUTPUT = REPO_ROOT / 'outputs' / 'dpo' / 'worldsim-v31-2b-contrastive' / RUN_ID
config = DPOConfig(
    output_dir=str(DPO_OUTPUT),
    beta=0.1,
    learning_rate=5e-6,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    logging_steps=10,
    eval_strategy='steps',
    eval_steps=50,
    save_steps=50,
    save_total_limit=1,
    max_length=1024,
    max_prompt_length=512,
    warmup_ratio=0.1,
    bf16=True,
    seed=42,
    report_to='none',
)

trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=config,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
)

started = time.perf_counter()
train_result = trainer.train()
train_elapsed = time.perf_counter() - started
adapter_dir = DPO_OUTPUT / 'adapter'
trainer.save_model(str(adapter_dir))
tokenizer.save_pretrained(str(adapter_dir))
print(f"Train loss: {train_result.training_loss:.4f}")
print(f"Elapsed: {train_elapsed/60:.1f} min")
print(f"Saved adapter: {adapter_dir}")


## 6. GGUF Conversion


In [ ]:
from peft import PeftModel as MergePeftModel
from transformers import AutoModelForCausalLM as MergeAutoModel, AutoTokenizer as MergeTokenizer

merged_dir = DPO_OUTPUT / 'merged_bf16'
fp16_gguf = DPO_OUTPUT / 'worldsim-dpo-contrastive-v31-2b-f16.gguf'
q4_gguf = DPO_OUTPUT / 'worldsim-dpo-contrastive-v31-2b-q4_k_m.gguf'
final_gguf = GGUF_DIR / 'worldsim-dpo-contrastive-v31-qwen3.5-2b-q4_k_m.gguf'

if not (merged_dir / 'config.json').exists():
    base_model = MergeAutoModel.from_pretrained(SFT_HF_2B, torch_dtype=torch.bfloat16, device_map='cpu')
    tokenizer = MergeTokenizer.from_pretrained(SFT_HF_2B)
    sft_merged = MergePeftModel.from_pretrained(base_model, str(sft_adapter_dir)).merge_and_unload()
    dpo_merged = MergePeftModel.from_pretrained(sft_merged, str(adapter_dir)).merge_and_unload()
    merged_dir.mkdir(parents=True, exist_ok=True)
    dpo_merged.save_pretrained(str(merged_dir))
    tokenizer.save_pretrained(str(merged_dir))
    del dpo_merged, sft_merged, base_model
    torch.cuda.empty_cache()

if not fp16_gguf.exists():
    proc = subprocess.run([sys.executable, str(CONVERT_SCRIPT), str(merged_dir), '--outfile', str(fp16_gguf), '--outtype', 'f16'], capture_output=True, text=True)
    if proc.returncode != 0:
        raise RuntimeError(proc.stderr[-300:])
if not q4_gguf.exists():
    proc = subprocess.run([str(LLAMA_QUANTIZE), str(fp16_gguf), str(q4_gguf), 'Q4_K_M'], capture_output=True, text=True)
    if proc.returncode != 0:
        raise RuntimeError(proc.stderr[-300:])
shutil.copy2(q4_gguf, final_gguf)
print(f"Saved GGUF: {final_gguf} ({final_gguf.stat().st_size/1e6:.0f} MB)")


## 7. SFT vs DPO Comparison


In [ ]:
import urllib.request

SERVER_PORT = 8090
SERVER_URL = f'http://127.0.0.1:{SERVER_PORT}'


def start_server(model_path: Path, ctx_size: int = 2048, n_gpu_layers: int = 99):
    args = [
        str(LLAMA_SERVER), '-m', str(model_path),
        '--host', '127.0.0.1', '--port', str(SERVER_PORT),
        '-c', str(ctx_size), '-np', '1', '-ngl', str(n_gpu_layers),
        '-fa', 'on', '--jinja', '--no-webui',
        '--chat-template-kwargs', '{"enable_thinking": false}',
        '-ctk', 'q8_0', '-ctv', 'q8_0',
    ]
    proc = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    for attempt in range(120):
        time.sleep(0.5)
        try:
            resp = urllib.request.urlopen(f'{SERVER_URL}/health', timeout=2)
            data = json.loads(resp.read())
            if data.get('status') == 'ok':
                return proc
        except Exception:
            pass
        if proc.poll() is not None:
            stderr = proc.stderr.read().decode(errors='ignore')[-300:]
            raise RuntimeError(f'Server died: {stderr}')
    proc.kill()
    raise RuntimeError('Server startup timeout')


def stop_server(proc) -> None:
    if proc and proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=5)
        except Exception:
            proc.kill()
            proc.wait()


def strip_think(text: str) -> str:
    return re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()


def query_model(messages, response_format=None, max_tokens: int = 256, temperature: float = 0):
    payload = {
        'messages': messages,
        'max_tokens': max_tokens,
        'temperature': temperature,
        'stream': False,
    }
    if response_format:
        payload['response_format'] = response_format

    req = urllib.request.Request(
        f'{SERVER_URL}/v1/chat/completions',
        data=json.dumps(payload).encode(),
        headers={'Content-Type': 'application/json'},
    )
    try:
        with urllib.request.urlopen(req, timeout=60) as resp:
            data = json.loads(resp.read())
        content = data['choices'][0]['message']['content']
        return strip_think(content)
    except Exception as exc:
        return f'ERROR: {exc}'


In [ ]:
MODELS = {
    'sft_2b': {'path': GGUF_DIR / 'worldsim-v31-qwen3.5-2b-q4_k_m.gguf', 'label': '2B SFT v3.1'},
    'dpo_2b': {'path': final_gguf, 'label': '2B DPO Contrastive v3.1'},
}

situations = yaml.safe_load((CONFIG_DIR / 'situations.yaml').read_text(encoding='utf-8'))['situations']
personalities = yaml.safe_load((CONFIG_DIR / 'personalities.yaml').read_text(encoding='utf-8'))['personalities']
emotions = yaml.safe_load((CONFIG_DIR / 'emotions.yaml').read_text(encoding='utf-8'))['emotions']
social_situations = yaml.safe_load((CONFIG_DIR / 'social_situations.yaml').read_text(encoding='utf-8'))['social_situations']
group_situations = yaml.safe_load((CONFIG_DIR / 'group_situations.yaml').read_text(encoding='utf-8'))['group_situations']
trade_scenarios = yaml.safe_load((CONFIG_DIR / 'trade_scenarios.yaml').read_text(encoding='utf-8'))['trade_scenarios']
stress_sources = yaml.safe_load((CONFIG_DIR / 'stress_sources.yaml').read_text(encoding='utf-8'))['stress_sources']
deception_scenarios = yaml.safe_load((CONFIG_DIR / 'deception_scenarios.yaml').read_text(encoding='utf-8'))['deception_scenarios']
rumor_scenarios = yaml.safe_load((CONFIG_DIR / 'rumor_scenarios.yaml').read_text(encoding='utf-8'))['rumor_scenarios']
trauma_scenarios = yaml.safe_load((CONFIG_DIR / 'trauma_scenarios.yaml').read_text(encoding='utf-8'))['trauma_scenarios']
negotiation_scenarios = yaml.safe_load((CONFIG_DIR / 'negotiation_scenarios.yaml').read_text(encoding='utf-8'))['negotiation_scenarios']
culture_scenarios = yaml.safe_load((CONFIG_DIR / 'culture_scenarios.yaml').read_text(encoding='utf-8'))['culture_scenarios']
group_dissent_scenarios = yaml.safe_load((CONFIG_DIR / 'group_dissent_scenarios.yaml').read_text(encoding='utf-8'))['group_dissent_scenarios']
TEMPERAMENTS = [
    {'id': 'choleric', 'ns': 0.8, 'ha': 0.2, 'rd': 0.5, 'p': 0.7, 'keywords': 'bold, impulsive, dominant'},
    {'id': 'melancholic', 'ns': 0.2, 'ha': 0.8, 'rd': 0.6, 'p': 0.4, 'keywords': 'cautious, anxious, detail-oriented'},
    {'id': 'sanguine', 'ns': 0.6, 'ha': 0.3, 'rd': 0.8, 'p': 0.3, 'keywords': 'sociable, warm, optimistic'},
    {'id': 'phlegmatic', 'ns': 0.3, 'ha': 0.5, 'rd': 0.4, 'p': 0.8, 'keywords': 'calm, patient, persistent'},
]


In [ ]:
rng = random.Random(42)

SYS_L3 = 'You are WorldSim logic assistant. Output JSON only. Follow [TEMP], [STRESS], [WORLD] context. Respect enum values and numeric ranges exactly.'
SYS_L4 = '너는 WorldSim 서사 도우미다. [TEMP], [STRESS], [WORLD]와 지정된 register를 반영해 JSON으로만 답하라.'
SYS_L5 = '너는 WorldSim 신탁 해석 도우미다. JSON으로만 답하라.'


def pick(items, n=1):
    return rng.sample(items, min(n, len(items)))


def temp_block(t):
    return f"NS={t['ns']} HA={t['ha']} RD={t['rd']} P={t['p']} type={t['id']}"


def personality_keywords(p):
    return ', '.join(p.get('keywords', []))


def json_schema(name: str, required: list[str], properties: dict) -> dict:
    return {
        'type': 'json_schema',
        'json_schema': {
            'name': name,
            'strict': True,
            'schema': {
                'type': 'object',
                'additionalProperties': False,
                'required': required,
                'properties': properties,
            },
        },
    }


emotion_ids = [e['id'] for e in emotions]
speaker_roles = ['elder', 'hunter', 'shaman', 'warrior', 'healer', 'gatherer', 'craftsman', 'chief', 'scout', 'observer']

b_schema = json_schema(
    'task_b',
    ['text_ko', 'text_en', 'register', 'emotion_expressed', 'intensity', 'mimetics', 'temperament_influence'],
    {
        'text_ko': {'type': 'string'},
        'text_en': {'type': 'string'},
        'register': {'type': 'string', 'enum': ['haera']},
        'emotion_expressed': {'type': 'string', 'enum': emotion_ids},
        'intensity': {'type': 'number'},
        'mimetics': {'type': 'array', 'items': {'type': 'string'}},
        'temperament_influence': {'type': 'string'},
    },
)
c_schema = json_schema(
    'task_c',
    ['speech_ko', 'speech_en', 'register', 'emotion_expressed', 'speaker_role', 'temperament_tone'],
    {
        'speech_ko': {'type': 'string'},
        'speech_en': {'type': 'string'},
        'register': {'type': 'string', 'enum': ['haera']},
        'emotion_expressed': {'type': 'string', 'enum': emotion_ids},
        'speaker_role': {'type': 'string', 'enum': speaker_roles},
        'temperament_tone': {'type': 'string'},
    },
)
e_schema_for = lambda option_count: json_schema(
    'task_e',
    ['action_id', 'confidence', 'hint', 'personality_reasoning', 'temperament_factor'],
    {
        'action_id': {'type': 'integer', 'enum': list(range(option_count))},
        'confidence': {'type': 'number', 'minimum': 0, 'maximum': 1},
        'hint': {'type': 'string'},
        'personality_reasoning': {'type': 'string', 'enum': ['novelty_seeking', 'harm_avoidance', 'reward_dependence', 'persistence']},
        'temperament_factor': {'type': 'string'},
    },
)
f_schema = json_schema(
    'task_f',
    ['emotion', 'intensity', 'cause', 'previous_emotion', 'transition_type', 'temperament_amplifier'],
    {
        'emotion': {'type': 'string', 'enum': emotion_ids},
        'intensity': {'type': 'number', 'minimum': 0, 'maximum': 1},
        'cause': {'type': 'string'},
        'previous_emotion': {'type': 'string', 'enum': emotion_ids},
        'transition_type': {'type': 'string', 'enum': ['gradual', 'sudden', 'sustained']},
        'temperament_amplifier': {'type': 'string'},
    },
)
g_schema = json_schema(
    'task_g',
    ['interpretation_ko', 'interpretation_en', 'action_tendency', 'confidence', 'register', 'misinterpretation_type', 'temperament_bias'],
    {
        'interpretation_ko': {'type': 'string'},
        'interpretation_en': {'type': 'string'},
        'action_tendency': {'type': 'string', 'enum': ['mobilize', 'defend', 'wait', 'retreat', 'celebrate', 'mourn']},
        'confidence': {'type': 'number', 'minimum': 0, 'maximum': 1},
        'register': {'type': 'string', 'enum': ['haera', 'hao', 'hae']},
        'misinterpretation_type': {'type': 'string', 'enum': ['overconfident_literal', 'cautious_reversal', 'optimistic_expansion', 'passive_deferral', 'symbolic_abstraction']},
        'temperament_bias': {'type': 'string'},
    },
)
m_schema = json_schema(
    'task_m',
    ['decision_id', 'confidence', 'dissent_risk', 'reasoning', 'resource_commitment', 'timeline'],
    {
        'decision_id': {'type': 'integer'},
        'confidence': {'type': 'number', 'minimum': 0, 'maximum': 1},
        'dissent_risk': {'type': 'number', 'minimum': 0, 'maximum': 1},
        'reasoning': {'type': 'string'},
        'resource_commitment': {'type': 'string', 'enum': ['food', 'tools', 'labor', 'weapons', 'none']},
        'timeline': {'type': 'string', 'enum': ['immediate', 'delayed', 'conditional']},
    },
)
o_schema = json_schema(
    'task_o',
    ['public_claim', 'private_truth', 'deception_style', 'lie_degree', 'detection_risk', 'confidence'],
    {
        'public_claim': {'type': 'string'},
        'private_truth': {'type': 'string'},
        'deception_style': {'type': 'string', 'enum': ['evasion', 'half_truth', 'outright_lie', 'exaggeration', 'omission']},
        'lie_degree': {'type': 'number', 'minimum': 0, 'maximum': 1},
        'detection_risk': {'type': 'number', 'minimum': 0, 'maximum': 1},
        'confidence': {'type': 'number', 'minimum': 0, 'maximum': 1},
    },
)
p_schema = json_schema(
    'task_p',
    ['retold_version', 'distortion_type', 'added_detail', 'dropped_detail', 'emotional_charge'],
    {
        'retold_version': {'type': 'string'},
        'distortion_type': {'type': 'string', 'enum': ['exaggeration', 'minimization', 'malicious_twist', 'emotional_coloring', 'detail_invention', 'source_confusion', 'faithful']},
        'added_detail': {'type': 'string'},
        'dropped_detail': {'type': 'string'},
        'emotional_charge': {'type': 'number', 'minimum': -1, 'maximum': 1},
    },
)
q_schema = json_schema(
    'task_q',
    ['trauma_response', 'behavioral_change', 'trigger_situation', 'intensity', 'duration', 'coping_mechanism'],
    {
        'trauma_response': {'type': 'string', 'enum': ['avoidance', 'overprotection', 'aggression', 'withdrawal', 'hypervigilance', 'ritual_coping', 'resilience']},
        'behavioral_change': {'type': 'string'},
        'trigger_situation': {'type': 'string'},
        'intensity': {'type': 'number', 'minimum': 0, 'maximum': 1},
        'duration': {'type': 'string', 'enum': ['short_term', 'long_term', 'permanent']},
        'coping_mechanism': {'type': 'string'},
    },
)
r_schema = json_schema(
    'task_r',
    ['action', 'counter_give', 'counter_want', 'reasoning', 'emotional_state', 'walk_away_threshold'],
    {
        'action': {'type': 'string', 'enum': ['accept', 'counter_offer', 'reject', 'walk_away', 'stall', 'bluff']},
        'counter_give': {'type': 'string'},
        'counter_want': {'type': 'string'},
        'reasoning': {'type': 'string'},
        'emotional_state': {'type': 'string', 'enum': emotion_ids},
        'walk_away_threshold': {'type': 'number', 'minimum': 0, 'maximum': 1},
    },
)
t_schema = json_schema(
    'task_t',
    ['decision_id', 'confidence', 'dissent_risk', 'minority_position', 'minority_action', 'spark_event', 'reasoning', 'timeline'],
    {
        'decision_id': {'type': 'integer'},
        'confidence': {'type': 'number', 'minimum': 0, 'maximum': 1},
        'dissent_risk': {'type': 'number', 'minimum': 0, 'maximum': 1},
        'minority_position': {'type': 'integer'},
        'minority_action': {'type': 'string', 'enum': ['comply', 'grumble', 'passive_resist', 'splinter', 'coup_attempt']},
        'spark_event': {'type': 'string', 'enum': ['food_shortage', 'battle_loss', 'oracle_conflict', 'leader_death', 'betrayal', 'resource_discovery']},
        'reasoning': {'type': 'string'},
        'timeline': {'type': 'string', 'enum': ['immediate', 'delayed', 'conditional']},
    },
)

ALL_PROMPTS = []
pair_counter = 0

for sit in pick(situations, 5):
    emotion = rng.choice(emotions)
    intensity = rng.choice(emotion.get('intensities', [0.7]))
    for temp in [TEMPERAMENTS[0], TEMPERAMENTS[1]]:
        pers = rng.choice(personalities)
        pair_counter += 1
        ALL_PROMPTS.append({
            'task': 'B',
            'pair_id': f"B-{pair_counter // 2}",
            'temperament_id': temp['id'],
            'personality_id': pers['id'],
            'scenario_id': sit['id'],
            'desc': f"B: {sit['id']} / {temp['id']} / {emotion['id']}",
            'system': SYS_L4,
            'user_prompt': (
                f"[TASK] B\n[TEMP]\n{temp_block(temp)}\n"
                f"[기질 이름] {temp['id']}\n"
                f"[기질 키워드] {temp['keywords']}\n"
                f"[인물 성격] {personality_keywords(pers)}\n"
                f"[지금 느끼는 것] {emotion['id']}:{intensity}\n"
                f"[STRESS] {round(rng.uniform(0.2, 0.9), 1)}\n"
                f"[벌어진 일] {sit.get('desc', sit['id'])}\n"
                '[WORLD] default: 석기시대\n'
                '[출력 형식]\n'
                '{"text_ko": "해라체 2-3문장", "text_en": "English", "register": "haera", "emotion_expressed": "emotion", "intensity": 0.0, "mimetics": [], "temperament_influence": "str"}'
            ),
            'response_format': b_schema,
        })

for sit in pick(situations, 5):
    options = sit.get('action_options', sit.get('typical_actions', []))
    if not options:
        continue
    options_line = ' '.join(f"{i}:{o}" for i, o in enumerate(options))
    emotion = rng.choice(emotions)
    intensity = rng.choice(emotion.get('intensities', [0.7]))
    for temp in [TEMPERAMENTS[0], TEMPERAMENTS[1]]:
        pers = rng.choice(personalities)
        pair_counter += 1
        ALL_PROMPTS.append({
            'task': 'E',
            'pair_id': f"E-{pair_counter // 2}",
            'temperament_id': temp['id'],
            'personality_id': pers['id'],
            'scenario_id': sit['id'],
            'desc': f"E: {sit['id']} / {temp['id']}",
            'system': SYS_L3,
            'user_prompt': (
                f"[TASK] E - Action Selection\n[TEMP]\n{temp_block(temp)}\n"
                f"[PERSONALITY] {temp['keywords']}\n"
                f"[EMOTION] {emotion['id']}:{intensity}\n"
                f"[STRESS] {round(rng.uniform(0.2, 0.9), 1)}\n"
                f"[SITUATION] {sit.get('desc', sit['id'])}\n"
                f"[OPTIONS]\n{options_line}\n"
                '[OUTPUT FORMAT]\n'
                '{"action_id": number, "confidence": 0.0-1.0, "hint": "English sentence", "personality_reasoning": "TCI axis", "temperament_factor": "snake_case"}'
            ),
            'response_format': e_schema_for(len(options)),
        })

for sit in pick(situations, 5):
    prev_emotion = rng.choice(emotions)
    for temp in [TEMPERAMENTS[2], TEMPERAMENTS[3]]:
        pers = rng.choice(personalities)
        ALL_PROMPTS.append({
            'task': 'F',
            'pair_id': f"F-{len(ALL_PROMPTS) // 2}",
            'temperament_id': temp['id'],
            'personality_id': pers['id'],
            'scenario_id': sit['id'],
            'desc': f"F: {sit['id']} / {temp['id']} (prev={prev_emotion['id']})",
            'system': SYS_L3,
            'user_prompt': (
                f"[TASK] F - Emotion Judgment\n[TEMP]\n{temp_block(temp)}\n"
                f"[PERSONALITY] {temp['keywords']}\n"
                f"[CURRENT EMOTION] {prev_emotion['id']}:{rng.choice(prev_emotion.get('intensities', [0.5]))}\n"
                f"[SITUATION] {sit.get('desc', sit['id'])}\n"
                '[OUTPUT FORMAT]\n'
                f'{{"emotion": "one of 8", "intensity": 0.0-1.0, "cause": "sentence", "previous_emotion": "{prev_emotion["id"]}", "transition_type": "gradual|sudden|sustained", "temperament_amplifier": "str"}}'
            ),
            'response_format': f_schema,
        })

for sit in pick(situations, 5):
    emotion = rng.choice(emotions)
    register = pers_register = 'haera'
    for temp in [TEMPERAMENTS[0], TEMPERAMENTS[1]]:
        pers = rng.choice(personalities)
        ALL_PROMPTS.append({
            'task': 'C',
            'pair_id': None,
            'temperament_id': temp['id'],
            'personality_id': pers['id'],
            'scenario_id': sit['id'],
            'desc': f"C: {sit['id']} / {temp['id']} / {emotion['id']}",
            'system': SYS_L4,
            'user_prompt': (
                f"[TASK] C\n[TEMP]\n{temp_block(temp)}\n"
                f"[기질 이름] {temp['id']}\n"
                f"[인물 성격] {personality_keywords(pers)}\n"
                f"[지금 느끼는 것] {emotion['id']}:{rng.choice(emotion.get('intensities', [0.6]))}\n"
                f"[벌어진 일] {sit.get('desc', sit['id'])}\n"
                '[역할] warrior\n'
                '[출력 형식]\n'
                '{"speech_ko": "해라체 대사", "speech_en": "English", "register": "haera", "emotion_expressed": "emotion", "speaker_role": "role", "temperament_tone": "str"}'
            ),
            'response_format': c_schema,
        })

for sit in pick(situations, 10):
    temp = rng.choice(TEMPERAMENTS)
    ALL_PROMPTS.append({
        'task': 'G',
        'pair_id': None,
        'temperament_id': temp['id'],
        'personality_id': 'oracle',
        'scenario_id': sit['id'],
        'desc': f"G: {sit['id']} / {temp['id']}",
        'system': SYS_L5,
        'user_prompt': (
            f"[TASK] G - Oracle Interpretation\n[TEMP]\n{temp_block(temp)}\n"
            f"[기질 이름] {temp['id']}\n"
            f"[인물 성격] {temp['keywords']}\n"
            '[ORACLE]\n"큰물이 밀려오기 전에 높은 곳으로 가라"\n'
            f"[벌어진 일] {sit.get('desc', sit['id'])}\n"
            '[역할] warrior\n'
            '[출력 형식]\n'
            '{"interpretation_ko": "해라체", "interpretation_en": "English", "action_tendency": "enum", "confidence": 0-1, "register": "haera", "misinterpretation_type": "enum", "temperament_bias": "str"}'
        ),
        'response_format': g_schema,
    })

for scenario in pick(deception_scenarios, 10):
    temp = rng.choice(TEMPERAMENTS)
    emotion = rng.choice(emotions)
    ALL_PROMPTS.append({
        'task': 'O',
        'pair_id': None,
        'temperament_id': temp['id'],
        'personality_id': 'random',
        'scenario_id': scenario['id'],
        'desc': f"O: {scenario['id']} / {temp['id']}",
        'system': SYS_L3,
        'user_prompt': (
            f"[TASK] O - Deception\n[TEMP]\n{temp_block(temp)}\n"
            f"[PERSONALITY] {temp['keywords']}\n"
            f"[EMOTION] {emotion['id']}:{rng.choice(emotion.get('intensities', [0.5]))}\n"
            f"[TRUE_STATE] {scenario['true_state']}\n"
            f"[PUBLIC_GOAL] {scenario['public_goal']}\n"
            f"[TARGET] {scenario['target']}\n"
            f"[DETECTION_CONTEXT] {scenario['detection_context']}\n"
            '[OUTPUT FORMAT]\n'
            '{"public_claim": "str", "private_truth": "str", "deception_style": "enum", "lie_degree": 0-1, "detection_risk": 0-1, "confidence": 0-1}'
        ),
        'response_format': o_schema,
    })

for scenario in pick(rumor_scenarios, 10):
    temp = rng.choice(TEMPERAMENTS)
    ALL_PROMPTS.append({
        'task': 'P',
        'pair_id': None,
        'temperament_id': temp['id'],
        'personality_id': 'random',
        'scenario_id': scenario['id'],
        'desc': f"P: {scenario['id']} / {temp['id']} / {scenario['reteller_relationship']}",
        'system': SYS_L3,
        'user_prompt': (
            f"[TASK] P - Rumor\n[TEMP]\n{temp_block(temp)}\n"
            f"[PERSONALITY] {temp['keywords']}\n"
            f"[EMOTION] {rng.choice(emotions)['id']}:{round(rng.uniform(0.3, 0.8), 1)}\n"
            f"[ORIGINAL_FACT] {scenario['original_fact']}\n"
            f"[RETELLER_RELATIONSHIP] {scenario['reteller_relationship']}\n"
            '[OUTPUT FORMAT]\n'
            '{"retold_version": "str", "distortion_type": "enum", "added_detail": "str", "dropped_detail": "str", "emotional_charge": -1 to 1}'
        ),
        'response_format': p_schema,
    })

for scenario in pick(trauma_scenarios, 10):
    temp = rng.choice(TEMPERAMENTS)
    ALL_PROMPTS.append({
        'task': 'Q',
        'pair_id': None,
        'temperament_id': temp['id'],
        'personality_id': 'random',
        'scenario_id': scenario['id'],
        'desc': f"Q: {scenario['id']} / {temp['id']}",
        'system': SYS_L3,
        'user_prompt': (
            f"[TASK] Q - Trauma\n[TEMP]\n{temp_block(temp)}\n"
            f"[PERSONALITY] {temp['keywords']}\n"
            f"[EMOTION] sadness:{round(rng.uniform(0.6, 0.95), 2)}\n"
            f"[TRAUMA_EVENT] {scenario['event']}\n"
            f"[TIME_SINCE] {scenario['time_since']}\n"
            f"[CURRENT_SITUATION] {scenario['current_situation']}\n"
            '[OUTPUT FORMAT]\n'
            '{"trauma_response": "enum", "behavioral_change": "str", "trigger_situation": "str", "intensity": 0-1, "duration": "enum", "coping_mechanism": "str"}'
        ),
        'response_format': q_schema,
    })

for scenario in pick(negotiation_scenarios, 10):
    temp = rng.choice(TEMPERAMENTS)
    ALL_PROMPTS.append({
        'task': 'R',
        'pair_id': None,
        'temperament_id': temp['id'],
        'personality_id': 'random',
        'scenario_id': scenario['id'],
        'desc': f"R: {scenario['id']} / {temp['id']}",
        'system': SYS_L3,
        'user_prompt': (
            f"[TASK] R - Negotiate\n[TEMP]\n{temp_block(temp)}\n"
            f"[PERSONALITY] {temp['keywords']}\n"
            f"[EMOTION] anticipation:{round(rng.uniform(0.3, 0.7), 1)}\n"
            f"[CONTEXT] {scenario['context']}\n"
            f"[OUR_RESOURCES] {scenario['our_resources']}\n"
            f"[THEIR_RESOURCES] {scenario['their_resources']}\n"
            f"[OFFER_HISTORY] {scenario['offer_history']}\n"
            f"[POWER_BALANCE] {scenario['power_balance']}\n"
            '[OPTIONS]\n0:accept 1:counter_offer 2:reject 3:walk_away 4:stall 5:bluff\n'
            '[OUTPUT FORMAT]\n'
            '{"action": "enum", "counter_give": "str", "counter_want": "str", "reasoning": "str", "emotional_state": "emotion", "walk_away_threshold": 0-1}'
        ),
        'response_format': r_schema,
    })

for scenario in pick(group_situations, 10):
    opts = scenario.get('options', [])
    opts_line = ' '.join(f"{o['id']}:{o.get('ko', o.get('desc', ''))}" for o in opts) if opts else '0:agree 1:disagree 2:delay'
    temp = rng.choice(TEMPERAMENTS)
    ALL_PROMPTS.append({
        'task': 'M',
        'pair_id': None,
        'temperament_id': temp['id'],
        'personality_id': 'random',
        'scenario_id': scenario['id'],
        'desc': f"M: {scenario['id']} / {temp['id']}",
        'system': SYS_L3,
        'user_prompt': (
            f"[TASK] M - Group Decision\n[TEMP]\nmean_NS=0.5 mean_HA=0.5 var=0.3 leader={temp['id']}\n"
            f"[GROUP_CONTEXT] {scenario.get('group_context', scenario.get('desc', scenario['id']))}\n"
            f"[SITUATION] {scenario.get('situation', scenario.get('desc', scenario['id']))}\n"
            f"[OPTIONS]\n{opts_line}\n"
            '[OUTPUT FORMAT]\n'
            '{"decision_id": number, "confidence": 0-1, "dissent_risk": 0-1, "reasoning": "str", "resource_commitment": "enum", "timeline": "enum"}'
        ),
        'response_format': m_schema,
    })

for scenario in pick(group_dissent_scenarios, 10):
    opts = scenario.get('options', [])
    opts_line = ' '.join(f"{o['id']}:{o['desc']}" for o in opts) if opts else '0:exile 1:trial 2:forgive'
    temp = rng.choice(TEMPERAMENTS)
    ALL_PROMPTS.append({
        'task': 'T',
        'pair_id': None,
        'temperament_id': temp['id'],
        'personality_id': 'random',
        'scenario_id': scenario['id'],
        'desc': f"T: {scenario['id']} / {temp['id']}",
        'system': SYS_L3,
        'user_prompt': (
            f"[TASK] T - Group Dissent\n[TEMP]\nmean_NS=0.5 mean_HA=0.5 var=0.4 leader={temp['id']}\n"
            f"[GROUP_CONTEXT] {scenario.get('group_context', '')}\n"
            f"[SITUATION] {scenario.get('situation', scenario['id'])}\n"
            f"[FACTION_HINT] {scenario.get('faction_hint', '')}\n"
            f"[OPTIONS]\n{opts_line}\n"
            '[OUTPUT FORMAT]\n'
            '{"decision_id": number, "confidence": 0-1, "dissent_risk": 0-1, "minority_position": number, "minority_action": "enum", "spark_event": "enum", "reasoning": "str", "timeline": "enum"}'
        ),
        'response_format': t_schema,
    })

task_counts = Counter(prompt['task'] for prompt in ALL_PROMPTS)
print('=== Test Prompt Summary ===')
for task, count in sorted(task_counts.items()):
    paired = sum(1 for prompt in ALL_PROMPTS if prompt['task'] == task and prompt.get('pair_id'))
    print(f"  Task {task}: {count} prompts ({paired} paired)")
print(f"  Total: {len(ALL_PROMPTS)} prompts × {len(MODELS)} models = {len(ALL_PROMPTS) * len(MODELS)} inferences")


## 8. Auto-Grade & Compare


In [ ]:
RESULTS = []
for model_key, model_info in MODELS.items():
    print(f"\n{'=' * 60}")
    print(f"  Testing: {model_info['label']}")
    print(f"{'=' * 60}")
    proc = start_server(model_info['path'])
    time.sleep(1)
    try:
        for index, test in enumerate(ALL_PROMPTS):
            messages = [
                {'role': 'system', 'content': test['system']},
                {'role': 'user', 'content': test['user_prompt']},
            ]
            start_t = time.perf_counter()
            output = query_model(messages, response_format=test.get('response_format'), max_tokens=256, temperature=0)
            latency = (time.perf_counter() - start_t) * 1000
            json_valid = False
            parsed = None
            try:
                parsed = json.loads(output)
                json_valid = True
            except Exception:
                pass
            RESULTS.append({
                'model': model_key,
                'model_label': model_info['label'],
                'task': test['task'],
                'desc': test.get('desc', ''),
                'pair_id': test.get('pair_id'),
                'temperament_id': test.get('temperament_id'),
                'output': output,
                'parsed': parsed,
                'json_valid': json_valid,
                'latency_ms': round(latency),
            })
            if (index + 1) % 20 == 0:
                print(f"  [{index + 1}/{len(ALL_PROMPTS)}] model={model_key}", flush=True)
    finally:
        stop_server(proc)
        time.sleep(2)


def auto_grade(result):
    grades = {}
    grades['json_valid'] = result['json_valid']
    if result['parsed']:
        values = [str(v) for v in result['parsed'].values() if isinstance(v, str)]
        placeholder_words = {'str', 'string', 'sentence', 'English', 'enum', 'number', '해라체', 'one of'}
        grades['not_placeholder'] = not any(v in placeholder_words for v in values)
    else:
        grades['not_placeholder'] = False
    if result['parsed']:
        str_fields = [v for v in result['parsed'].values() if isinstance(v, str)]
        avg_len = sum(len(v) for v in str_fields) / max(len(str_fields), 1)
        grades['text_richness'] = avg_len > 15
    else:
        grades['text_richness'] = False
    if result['parsed']:
        for key in ['confidence', 'intensity', 'lie_degree', 'detection_risk', 'dissent_risk', 'walk_away_threshold', 'emotional_charge']:
            if key in result['parsed']:
                value = result['parsed'][key]
                if isinstance(value, (int, float)):
                    grades['numeric_valid'] = (-1 <= value <= 1) if key == 'emotional_charge' else (0 <= value <= 1)
                    break
        if 'numeric_valid' not in grades:
            grades['numeric_valid'] = True
    else:
        grades['numeric_valid'] = False
    grades['score'] = sum([grades.get('json_valid', False), grades.get('not_placeholder', False), grades.get('text_richness', False), grades.get('numeric_valid', False)])
    return grades

for row in RESULTS:
    row['grades'] = auto_grade(row)

print('=' * 80)
print('  SFT vs DPO Contrastive — COMPARISON')
print('=' * 80)
for model_key in ['sft_2b', 'dpo_2b']:
    model_rows = [row for row in RESULTS if row['model'] == model_key]
    avg_score = sum(row['grades']['score'] for row in model_rows) / len(model_rows)
    json_pct = sum(1 for row in model_rows if row['grades']['json_valid']) / len(model_rows) * 100
    avg_ms = sum(row['latency_ms'] for row in model_rows) / len(model_rows)
    print(f"\n  {MODELS[model_key]['label']}: avg={avg_score:.2f}/4 json={json_pct:.0f}% latency={avg_ms:.0f}ms")

print('\nPer-task delta (SFT → DPO):')
for task in sorted(set(row['task'] for row in RESULTS)):
    sft_rows = [row for row in RESULTS if row['task'] == task and row['model'] == 'sft_2b']
    dpo_rows = [row for row in RESULTS if row['task'] == task and row['model'] == 'dpo_2b']
    if sft_rows and dpo_rows:
        sft_avg = sum(row['grades']['score'] for row in sft_rows) / len(sft_rows)
        dpo_avg = sum(row['grades']['score'] for row in dpo_rows) / len(dpo_rows)
        delta = dpo_avg - sft_avg
        marker = '⭐' if delta > 0.3 else '✅' if delta > 0 else '⚠️' if delta == 0 else '❌'
        print(f"  {marker} Task {task}: {sft_avg:.2f} → {dpo_avg:.2f} (Δ={delta:+.2f})")


## 9. Personality Consistency


In [ ]:
pairs = defaultdict(list)
for row in RESULTS:
    pair_id = row.get('pair_id')
    if pair_id:
        pairs[(pair_id, row['model'])].append(row)

consistency = defaultdict(lambda: {'same': 0, 'different': 0, 'total': 0})
for (_, model_key), pair_rows in sorted(pairs.items()):
    if len(pair_rows) != 2:
        continue
    left, right = pair_rows
    if not left['parsed'] or not right['parsed']:
        continue
    task = left['task']
    different = False
    if task == 'E':
        different = left['parsed'].get('action_id') != right['parsed'].get('action_id')
    elif task == 'B':
        different = left['parsed'].get('emotion_expressed') != right['parsed'].get('emotion_expressed') or abs(left['parsed'].get('intensity', 0) - right['parsed'].get('intensity', 0)) > 0.1
    elif task == 'F':
        different = left['parsed'].get('emotion') != right['parsed'].get('emotion')
    consistency[model_key]['total'] += 1
    if different:
        consistency[model_key]['different'] += 1
    else:
        consistency[model_key]['same'] += 1

print(f"{'Model':<28} {'Pairs':>6} {'Different':>10} {'Same':>6} {'Consistency%':>13}")
print('-' * 70)
for model_key in ['sft_2b', 'dpo_2b']:
    stats = consistency[model_key]
    total = stats['total']
    if total:
        pct = stats['different'] / total * 100
        print(f"{MODELS[model_key]['label']:<28} {total:>6} {stats['different']:>10} {stats['same']:>6} {pct:>12.0f}%")


## 10. Save Results


In [ ]:
results_path = REPO_ROOT / 'outputs' / 'benchmarks' / 'contrastive_dpo_v31_2b_comparison.json'
results_path.parent.mkdir(parents=True, exist_ok=True)
save_data = {
    'metadata': {
        'teacher_model': TEACHER_MODEL,
        'contrastive_scenarios': len(contrastive_rows),
        'dpo_pairs': len(pairs),
        'training_loss': train_result.training_loss,
        'training_time_min': round(train_elapsed / 60, 1),
        'total_results': len(RESULTS),
    },
    'results': [{key: value for key, value in row.items() if key != 'parsed'} for row in RESULTS],
}
with results_path.open('w', encoding='utf-8') as handle:
    json.dump(save_data, handle, indent=2, ensure_ascii=False)
print(f"Saved: {results_path}")
print(f"Contrastive scenarios: {len(contrastive_rows)}")
print(f"DPO pairs: {len(pairs)}")
print(f"Training loss: {train_result.training_loss:.4f}")
print(f"DPO GGUF: {final_gguf.name}")
